## Cambios en models

In [ ]:
from django.db import models

# Create your models here.

class ProductModel(models.Model):
    title = models.TextField()
    price = models.FloatField()
    description = models.TextField(null=True)
    pieces = models.IntegerField(default="1")
    lego_points = models.IntegerField(default="1")
    set_id = models.IntegerField(default="1")

In [ ]:
python manage.py makemigrations
python manage.py migrate

## Borrar campo en models

In [ ]:
from django.db import models

# Create your models here.

class ProductModel(models.Model):
    title = models.TextField()
    price = models.FloatField()
    description = models.TextField(null=True)
    pieces = models.IntegerField(default="1")
    lego_points = models.IntegerField(default="1")


In [ ]:
python manage.py makemigrations
python manage.py migrate

## Borrar migraciones y Compresión en Migraciones

In [ ]:
python manage.py squashmigrations <APP_LABEL> <MIGRATION_NUMBER>

python manage.py squashmigrations ecommerce 0004

python manage.py migrate

## Django Shell - guardar data

python manage.py shell

In [ ]:
>>> from ecommerce.models import ProductModel

## Crear
>>> ProductModel.objects.create(title="Producto 1", price=199.99)

In [ ]:
Queryset (qs): lista de los objetos almacenados para cierto modelo

In [ ]:
queryset = ProductModel.objects.all()
qs = ProductModel.objects.all()

In [ ]:
qs.filter(title__icontains="producto")

In [ ]:
## Editar
my_product = ProductModel.objects.get(id=2)
>>> my_product.title
'Dinosaur Fossils: Tyrannosaurus rex'
>>> my_product.description
'Dinosaur Fossils: Tyrannosaurus rex'
>>> my_product.price
5999.0
>>> my_product.price = 5998.00
>>> my_product.save()

In [ ]:
## Borrar
>>> ProductModel.object.get(id=14).delete()

## Validación de campos en Modelos

### ecommerce/validators.py

In [ ]:
from django.core.exceptions import ValidationError

BLOCKED_WORDS = [
    "barato",
    "malo"
]

def validate_blocked_words(value):
    init_string = f"{value}".lower()
    unique_words = set(init_string.split())
    blocked_words = set(BLOCKED_WORDS)
    invalid_words = (unique_words & blocked_words)
    has_error = len(invalid_words) > 0
    if has_error:
        errors = []
        for invalid_word in invalid_words:
            msg = "{} es una palabra no permitida".format(invalid_word)
            errors.appends(msg)
        raise ValidationError(errors)
    return value

### ecommerce/models.py

In [ ]:
from django.db import models

from .validators import validate_blocked_words

# Create your models here.

class ProductModel(models.Model):
    title = models.TextField()
    price = models.FloatField()
    description = models.TextField(default="Not Specified")
    pieces = models.IntegerField(default="1")
    lego_points = models.IntegerField(default="1")
    set_id = models.IntegerField(null=True)
    
    def save(self, *args, **kwargs):
        validate_blocked_words(self.title)
        super().save(*args, **kwargs)

## Agregar opciones a los campos de modelos

In [ ]:
from django.db import models

from .validators import validate_blocked_words

# VALOR_EN_DB, VALOR_USUARIO

PUBLISH_STATE_CHOICES = [
    ("ER", "BORRADOR"),
    ("PU", "PUBLICADO"),
    ("PR", "PRIVADO")
]

# Create your models here.

class ProductModel(models.Model):
    state = models.CharField(max_length=2, choices=PUBLISH_STATE_CHOICES, default="BR")
    title = models.TextField()
    price = models.FloatField()
    description = models.TextField(default="Not Specified")
    pieces = models.IntegerField(default="1")
    lego_points = models.IntegerField(default="1")
    set_id = models.IntegerField(null=True)
    
    def save(self, *args, **kwargs):
        validate_blocked_words(self.title)
        super().save(*args, **kwargs)
    
    def is_published(self):
        return self.state == "PU"

## Agregar opciones avanzadas a Models

In [ ]:
from django.db import models

from .validators import validate_blocked_words

# Create your models here.

class ProductModel(models.Model):
    class ProductStateOptions(models.TextChoices):
        PUBLISHED = "PU", "PUBLICADO"
        DRAFT = "BR", "BORRADOR"
        PRIVATE = "PR", "PRIVADO"

    state = models.CharField(max_length=2, choices=ProductStateOptions.choices, default=ProductStateOptions.DRAFT)
    title = models.TextField()
    price = models.FloatField()
    description = models.TextField(default="Not Specified")
    pieces = models.IntegerField(default="1")
    lego_points = models.IntegerField(default="1")
    set_id = models.IntegerField(null=True)
    
    def save(self, *args, **kwargs):
        validate_blocked_words(self.title)
        super().save(*args, **kwargs)
    
    def is_published(self):
        return self.state == self.ProductStateOptions.PUBLISHED

## Modelo Abstracto como base

In [ ]:
# Create new app called base

python manage.py startapp base

### base/models

In [ ]:
from django.db import models
from django.utils import timezone

# Create your models here.

class BasePublishModel(models.Model):
    class PublishStateOptions(models.TextChoices):
        PUBLISHED = "PU", "PUBLICADO"
        DRAFT = "BR", "BORRADOR"
        PRIVATE = "PR", "PRIVADO"

    state = models.CharField(max_length=2, choices=PublishStateOptions.choices, default=PublishStateOptions.DRAFT)
    timestamp = models.DateTimeField(auto_now_add=True)
    updated = models.DateTimeField(auto_now_add=True)
    publish_timestamp = models.DateTimeField(auto_now_add=False, auto_now=False, null=True)
    
    class Meta:
        abstract = True
        ordering = ["-updated", "-timestamp"]

    def save(self, *args, **kwargs):
        if self.state_is_published and self.publish_timestamp is None:
            self.publish_timestamp = timezone.now()
        else:
            self.publish_timestamp = None
        super().save(*args, **kwargs)
    
    @property
    def state_is_published(self):
        return self.state == self.PublishStateOptions.PUBLISHED

    def is_published(self):
        publish_timestamp = self.publish_timestamp
        return self.state_is_published and publish_timestamp < timezone.now()

### ecommerce/models

In [ ]:
from django.db import models

from base.models import BasePublishModel
from .validators import validate_blocked_words

# Create your models here.

class ProductModel(BasePublishModel):
    
    title = models.TextField()
    price = models.FloatField()
    description = models.TextField(default="Not Specified")
    pieces = models.IntegerField(default="1")
    lego_points = models.IntegerField(default="1")
    set_id = models.IntegerField(null=True)
    
    def save(self, *args, **kwargs):
        validate_blocked_words(self.title)
        super().save(*args, **kwargs)

### config/settings

In [ ]:
...

INSTALLED_APPS = [
    "pages.apps.PagesConfig",
    "ecommerce.apps.EcommerceConfig",
    "base.apps.BaseConfig", # <--------
    "django.contrib.admin",
    "django.contrib.auth",
    "django.contrib.contenttypes",
    "django.contrib.sessions",
    "django.contrib.messages",
    "django.contrib.staticfiles",
]

...

In [ ]:
python manage.py makemigrations

python manage.py migrate

## Creación a Granel (Bulk creation)

In [ ]:
products_data = []
for i in range(1,10):
    new_data = {"title": "Producto {}".format(i), "price": i*100+99.99}
    products_data.append(new_data)

In [ ]:
from ecommerce.models import ProductModel
new_objects = []
for product_data in products_data:
    print(product_data)
    new_objects.append(ProductModel(**product_data))

In [ ]:
ProductModel.objects.bulk_create(new_objects, ignore_conflicts=True)

## SlugField y Señales en Modelos

slug = se usa para los URLS

title: escritorio con altura ajustable
slug: /escritorio-con-altura-ajustable
url: www.mywebsite.com/escritorio-con-altura-ajustable
url: www.mywebsite.com/1

Signals / Señales
- pre_save
- post_save
- pre_delete
- post_delete
- pre_init
- post_init
- pre_migrate
- post_migrate

In [ ]:
# Get signals directory

from django.db.models import signals
dir(signals)

### ecommerce/models.py

In [ ]:
from django.db import models
from django.db.models.signals import pre_save
from django.utils.text import slugify

from base.models import BasePublishModel
from .validators import validate_blocked_words

# Create your models here.

class ProductModel(BasePublishModel):
    
    title = models.TextField()
    price = models.FloatField()
    description = models.TextField(default="Not Specified")
    pieces = models.IntegerField(default="1")
    lego_points = models.IntegerField(default="1")
    set_id = models.IntegerField(default="1")
    slug = models.SlugField(null=True, blank=True, db_index=True)

    def get_absolute_url(self):
        return f"/product/{self.slug}"
    
    def save(self, *args, **kwargs):
        validate_blocked_words(self.title)
        super().save(*args, **kwargs)

def slugify_pre_save(sender, instance, *args, **kwargs):
    if instance.slug is None or instance.slug == "":
        new_slug = slugify(instance.title)
        MyModel = instance.__class__
        qs = MyModel.objects.filter(slug__startswith=new_slug).exclude(id=instance.id)
        if qs.count() == 0:
            instance.slug = new_slug
        else:
            instance.slug = f"{new_slug}-{qs.count()}"

pre_save.connect(slugify_pre_save, sender=ProductModel)

In [ ]:
# Shell

from ecommerce.models import ProductModel

obj1 = ProductModel.objects.create(title="Lego test slug", price=842.12)
obj1.slug

obj2 = ProductModel.objects.create(title="Lego test slug", price=842.12)
obj2.slug

# Fixtures para Cargar Data

In [ ]:
docker exec -it hellodjango-web-1 bash

In [ ]:
python manage.py dumpdata ecommerce --indent 4 --format json

In [ ]:
python manage.py dumpdata ecommerce --indent 4 --format json > ecommerce/fixtures/ProductModel.json

In [ ]:
# Shell (python manage.py shell)

from ecommerce.models import ProductModel

ProductModel.objects.all()

ProductModel.objects.all().delete()

In [ ]:
# bash

python manage.py loaddata ecommerce/fixtures/ProductModel.json

# Llaves foráneas en Modelos

In [ ]:
ecommerce/models.py

In [ ]:
from django.conf import settings <-----------
from django.db import models
from django.db.models.signals import pre_save
from django.utils.text import slugify

from base.models import BasePublishModel
from .validators import validate_blocked_words

User = settings.AUTH_USER_MODEL <------------

# Create your models here.

class ProductModel(BasePublishModel):
    
    title = models.TextField()
    price = models.FloatField()
    description = models.TextField(null=True)
    pieces = models.IntegerField(null=True)
    lego_points = models.IntegerField(null=True)
    set_id = models.IntegerField(null=True)
    slug = models.SlugField(null=True, blank=True, db_index=True)
    user = models.ForeignKey(User, null=True, on_delete=models.SET_NULL) <----------------

In [ ]:
python manage.py makemigrations
python manage.py migrate